In [ ]:
!git clone https://github.com/kethansplunk/Codegen.git
%cd Codegen
!pip install -q torch transformers peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv

Cloning into 'Codegen'...
remote: Enumerating objects: 1073, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 1073 (delta 78), reused 87 (delta 35), pack-reused 931 (from 1)
Receiving objects: 100% (1073/1073), 17.63 MiB | 29.41 MiB/s, done.
Resolving deltas: 100% (797/797), done.
/content/Codegen
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 144.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 19.4 MB/s eta 

In [ ]:
# Install and start mongod
!apt-get install -y mongodb >/dev/null 2>&1 || (curl -fsSL https://pgp.mongodb.com/server-7.0.asc | sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor && echo "deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list && apt-get update -qq && apt-get install -y mongodb-org)
!mkdir -p /data/db
import subprocess
subprocess.Popen(["mongod", "--dbpath", "/data/db", "--bind_ip", "127.0.0.1"])
import time; time.sleep(5)
!mongosh --eval "db.version()"   # confirm it's up


deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  mongodb-database-tools mongodb-mongosh mongodb-org-database
  mongodb-org-database-tools-extra mongodb-org-mongos mongodb-org-server
  mongodb-org-shell mongodb-org-tools
The following NEW packages will be installed:
  mongodb-database-tools mongodb-mongosh mongodb-org mongodb-org-database
  mongodb-org-database-tools-extra mongodb-org-mongos mongodb-org-server
  mongodb-org-shell mongodb-org-tools
0 upgraded, 9 newly installed, 0 to remove and 134 not upgraded.
Need to get 189 MB of archives.
After this operation,

In [ ]:
# 1. Mount Drive (skip if already mounted from the SQL setup earlier)
from google.colab import drive
drive.mount('/content/drive')

# 2. Install deps — pymongo added for the NoSQL/Mongo pieces
!pip install -q peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv pymongo

# 3. Symlink Drive checkpoints — NOTE: sources say _nosql, not _sql
import os
%cd /content/Codegen

DRIVE = '/content/drive/MyDrive/codegen'
os.makedirs('models', exist_ok=True)

# os.symlink(f'{DRIVE}/checkpoints/sar_nosql',       'models/sar_nosql')
# os.symlink(f'{DRIVE}/checkpoints/generator_nosql', 'models/generator_nosql')
os.symlink(f'{DRIVE}/checkpoints/sar_sql',       'models/sar_sql')
os.symlink(f'{DRIVE}/checkpoints/generator_sql', 'models/generator_sql')

# 4. Sanity check the RIGHT paths this time
!ls -la models/sar_nosql models/generator_nosql
# 4. Sanity check the symlinks actually resolve
!ls -la models/sar_sql models/generator_sql indexes


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp /content/drive/MyDrive/codegen/checkpoints/spider_database.zip /content/Codegen/
!unzip -q /content/Codegen/spider_database.zip -d /content/Codegen/Data/Spider/
!ls /content/Codegen/Data/Spider/database | wc -l
!ls /content/Codegen/Data/Spider/database | head -5


166
academic
activity_1
aircraft
allergy_1
apartment_rentals


In [ ]:
from src.mongodb_converter import MongoDBConverter

converter = MongoDBConverter()
for db_name in ["department_management", "concert_singer"]:  # adjust to what you want to test
    converter.convert_database(
        db_root="Data/Spider/database",
        db_name="concert_singer",
        fk_graph_dir="Data/fk_graphs",
    )
print("Loaded.")


Loaded.


In [ ]:
from src.mongodb_converter import convert_all

convert_all(
    db_root="Data/Spider/database",
    fk_graph_dir="Data/fk_graphs",
    schema_cache_dir="Data/mongodb",   # matches configs/config.yaml's mongodb.schema_cache
)


[1/166] academic — skipped (cached)
[2/166] activity_1 — skipped (cached)
[3/166] aircraft — skipped (cached)
[4/166] allergy_1 — skipped (cached)
[5/166] apartment_rentals — skipped (cached)
[6/166] architecture — skipped (cached)
[7/166] assets_maintenance — skipped (cached)
[8/166] baseball_1 — skipped (cached)
[9/166] battle_death — skipped (cached)
[10/166] behavior_monitoring — skipped (cached)
[11/166] bike_1 — skipped (cached)
[12/166] body_builder — skipped (cached)
[13/166] book_2 — skipped (cached)
[14/166] browser_web — skipped (cached)
[15/166] candidate_poll — skipped (cached)
[16/166] car_1 — skipped (cached)
[17/166] chinook_1 — skipped (cached)
[18/166] cinema — skipped (cached)
[19/166] city_record — skipped (cached)
[20/166] climbing — skipped (cached)
[21/166] club_1 — skipped (cached)
[22/166] coffee_shop — skipped (cached)
[23/166] college_1 — skipped (cached)
[24/166] college_2 — skipped (cached)
[25/166] college_3 — skipped (cached)
[26/166] company_1 — skipped 

In [ ]:
import shutil
shutil.rmtree('Data/mongodb', ignore_errors=True)

from src.mongodb_converter import convert_all
convert_all(
    db_root="Data/Spider/database",
    fk_graph_dir="Data/fk_graphs",
    schema_cache_dir="Data/mongodb",
)


[1/166] academic — 15 collections, 42 fields
[2/166] activity_1 — 5 collections, 22 fields
[3/166] aircraft — 5 collections, 28 fields
[4/166] allergy_1 — 3 collections, 12 fields
[5/166] apartment_rentals — 6 collections, 31 fields
[6/166] architecture — 3 collections, 17 fields
[7/166] assets_maintenance — 14 collections, 64 fields
[8/166] baseball_1 — 26 collections, 352 fields
[9/166] battle_death — 3 collections, 18 fields
[10/166] behavior_monitoring — 11 collections, 64 fields
[11/166] bike_1 — 4 collections, 46 fields
[12/166] body_builder — 2 collections, 11 fields
[13/166] book_2 — 2 collections, 9 fields
[14/166] browser_web — 3 collections, 11 fields
[15/166] candidate_poll — 2 collections, 14 fields
[16/166] car_1 — 6 collections, 23 fields
[17/166] chinook_1 — 11 collections, 64 fields
[18/166] cinema — 3 collections, 17 fields
[19/166] city_record — 4 collections, 27 fields
[20/166] climbing — 2 collections, 12 fields
[21/166] club_1 — 3 collections, 15 fields
[22/166] c

In [ ]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
print(len(client.list_database_names()), client.list_database_names()[:10])
print(client['formula_1']['drivers'].count_documents({}))   # must be > 0


169 ['academic', 'activity_1', 'admin', 'aircraft', 'allergy_1', 'apartment_rentals', 'architecture', 'assets_maintenance', 'baseball_1', 'battle_death']
842


In [ ]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
print(client.list_database_names())


['admin', 'concert_singer', 'config', 'local']


In [ ]:
!python -m scripts.run_posg_nosql --smoke_test --n 10 --hard


Running on: cuda
Data source: Data/cot_data/nosql_cot_train.json
--hard: filtered 5410 -> 4015 $lookup/$group/multi-stage entries
Mongo URI: mongodb://localhost:27017  (make sure mongod is running and target DBs are loaded)
Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2723.73it/s]
Loaded corpus: 5697 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 23/23 [00:00<00:00, 201.42it/s]
Inference Embeddings: 100% 23/23 [00:00<00:00, 29.33it/s]
Loading Generator ...
config.json: 100% 663/663 [00:00<00:00, 4.14MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 55.1MB/s]
Fetching 4 files: 100% 4/4 [02:32<00:00, 38.14s/it] 
Download complete: 100% 15.2G/15.2G [02:32<00:00, 99.8MB/s]
Loading weights: 100% 339/339 [00:03<00:00, 87.69it/s] 
generation_config.json: 100% 242/242 [00:00<00:00, 1.68MB/s]

[1/10] Find the name and city of the airport which is the source for the most num

In [ ]:
!python -m scripts.run_posg_nosql --smoke_test --n 30 --hard

Running on: cuda
Data source: Data/cot_data/nosql_cot_train.json
--hard: filtered 5410 -> 4015 $lookup/$group/multi-stage entries
Mongo URI: mongodb://localhost:27017  (make sure mongod is running and target DBs are loaded)
Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 3009.78it/s]
Loaded corpus: 5697 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 23/23 [00:00<00:00, 210.72it/s]
Inference Embeddings: 100% 23/23 [00:00<00:00, 32.56it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 88.25it/s] 

[1/30] Find the name and city of the airport which is the source for the most number of flight routes.
  gold:     {"collection": "routes", "pipeline": [{"$group": {"_id": "$src_apid", "count": {"$sum": 1}}}, {"$sort": {"count": -1}}, {"$limit": 1}, {"$lookup": {"from": "airports", "localField": "_id", "foreignField": "apid", "as": "airport"}}, {"$unwind": "$airport"}, {"$p

In [ ]:
!python -m scripts.run_posg_nosql --smoke_test --n 30 --hard --seed 0

Running on: cuda
Data source: Data/cot_data/nosql_cot_train.json
--hard: filtered 5410 -> 4015 $lookup/$group/multi-stage entries
Mongo URI: mongodb://localhost:27017  (make sure mongod is running and target DBs are loaded)
Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2573.75it/s]
Loaded corpus: 5697 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 23/23 [00:00<00:00, 213.79it/s]
Inference Embeddings: 100% 23/23 [00:00<00:00, 32.59it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 88.69it/s]

[1/30] Find the name and city of the airport which is the source for the most number of flight routes.
  gold:     {"collection": "routes", "pipeline": [{"$group": {"_id": "$src_apid", "count": {"$sum": 1}}}, {"$sort": {"count": -1}}, {"$limit": 1}, {"$lookup": {"from": "airports", "localField": "_id", "foreignField": "apid", "as": "airport"}}, {"$unwind": "$airport"}, {"$pr

In [ ]:
!python -m scripts.run_posg_nosql --smoke_test --n 30


Running on: cuda
Data source: Data/cot_data/nosql_cot_train.json
Mongo URI: mongodb://localhost:27017  (make sure mongod is running and target DBs are loaded)
Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2774.08it/s]
Loaded corpus: 5697 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 23/23 [00:00<00:00, 197.49it/s]
Inference Embeddings: 100% 23/23 [00:00<00:00, 31.83it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 88.24it/s]

[1/30] show the titles, and authors or editors for all books made after the year 1989.
  gold:     {"collection": "book_club", "pipeline": [{"$match": {"Year": {"$gt": 1989}}}, {"$project": {"book_title": "$Book_Title", "author_or_editor": "$Author_or_Editor", "_id": 0}}]}
  greedy:   {"collection": "book_club", "pipeline": [{"$match": {"Year": {"$gt": 1989}}}, {"$project": {"book_title": "$Book_Title", "author_or_editor": "$Author_or_Edi

In [ ]:
with open('.env', 'w') as f:
    f.write('DEEPSEEK_API_KEY=your_actual_key_here\n')


In [ ]:
!python -m scripts.build_dev_eval_set --out Data/cot_data/sql_dev_eval_full.json

Dev entries: 1034
  20/1034 processed (20 written, 0 skipped: no cached schema)
  40/1034 processed (40 written, 0 skipped: no cached schema)
  60/1034 processed (60 written, 0 skipped: no cached schema)
  80/1034 processed (80 written, 0 skipped: no cached schema)
  100/1034 processed (100 written, 0 skipped: no cached schema)
  120/1034 processed (120 written, 0 skipped: no cached schema)
  140/1034 processed (140 written, 0 skipped: no cached schema)
  160/1034 processed (160 written, 0 skipped: no cached schema)
[SchemaLinker API] Parse failed on attempt 1/3
  180/1034 processed (180 written, 0 skipped: no cached schema)
  200/1034 processed (200 written, 0 skipped: no cached schema)
  220/1034 processed (220 written, 0 skipped: no cached schema)
  240/1034 processed (240 written, 0 skipped: no cached schema)
  260/1034 processed (260 written, 0 skipped: no cached schema)
  280/1034 processed (280 written, 0 skipped: no cached schema)
  300/1034 processed (300 written, 0 skipped: n

In [ ]:
!python -m scripts.run_posg_sql --smoke_test --n 30 --data Data/cot_data/sql_dev_eval_full.json


Running on: cuda
Data source: Data/cot_data/sql_dev_eval_full.json
Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2671.99it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/Codegen/scripts/run_posg_sql.py", line 299, in <module>
    main()
  File "/content/Codegen/scripts/run_posg_sql.py", line 288, in main
    smoke_test(config, n=args.n, strategy=args.strategy, seed=args.seed,
  File "/content/Codegen/scripts/run_posg_sql.py", line 154, in smoke_test
    sar = get_sar_retriever(config["sar"], track="sql")
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Codegen/src/sar/infer.py", line 244, in get_sar_retriever
    return SARRetriever(
           ^^^^^^^^^^^^^
  File "/content/Codegen/src/sar/infer.py", line 51, in __init__
    torch.load(model_path, map_location=self.device)
  File "/usr/local/lib/python3.12/dist-packages/torch/